In [11]:
from pyspark.sql import (
    functions as F,
    Window
)

In [1]:
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder.appName("test").getOrCreate()

24/10/20 19:27:59 WARN Utils: Your hostname, Deepaks-MacBook-Pro.local resolves to a loopback address: 127.0.0.1; using 192.168.1.44 instead (on interface en0)
24/10/20 19:27:59 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
24/10/20 19:27:59 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
rows = [
    [ 1, 1],
    [ 1, 4],
    [ 1, 2],
    [ 2, 4],
    [ 2, 5],
    [ 2, 4],
    [ 2, 5],
]

In [4]:
import pandas as pd

In [16]:
df = spark.createDataFrame(pd.DataFrame(rows).rename(columns={0:'a', 1:'b'}))

In [17]:
df = df.withColumn('tmp', F.lit(1))

In [20]:
window = Window.partitionBy('a').orderBy('b')

In [22]:
df = df.withColumn("incr", F.row_number().over(window))

In [29]:
user_requests_df = df.groupBy('a').count()

In [30]:
df = df.join(user_requests_df, on='a', how='inner')

In [31]:
df.show()

+---+---+---+----+-----+
|  a|  b|tmp|incr|count|
+---+---+---+----+-----+
|  1|  4|  1|   3|    3|
|  1|  2|  1|   2|    3|
|  1|  1|  1|   1|    3|
|  2|  5|  1|   4|    4|
|  2|  5|  1|   3|    4|
|  2|  4|  1|   2|    4|
|  2|  4|  1|   1|    4|
+---+---+---+----+-----+



In [36]:
df = df.withColumn("percentile", (F.col("incr"))/F.col('count'))

In [37]:
df.show()

+---+---+---+----+-----+------------------+
|  a|  b|tmp|incr|count|        percentile|
+---+---+---+----+-----+------------------+
|  1|  4|  1|   3|    3|               1.0|
|  1|  2|  1|   2|    3|0.6666666666666666|
|  1|  1|  1|   1|    3|0.3333333333333333|
|  2|  5|  1|   4|    4|               1.0|
|  2|  5|  1|   3|    4|              0.75|
|  2|  4|  1|   2|    4|               0.5|
|  2|  4|  1|   1|    4|              0.25|
+---+---+---+----+-----+------------------+



In [43]:
df.select((F.col('a') == 1) == True).show()

+----------------+
|((a = 1) = true)|
+----------------+
|            true|
|            true|
|            true|
|           false|
|           false|
|           false|
|           false|
+----------------+

